In [2]:
''' getting important libraries which is used during our Model building
    from xgboost  we imported XGBRegressor,
    and then we evaluted our model using various metrics from sklearn.metrics like
    r2_score,mean_absolute_error,mean_squared_Error'''
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error

In [3]:
'''making a DF pipeline to our cleaned data set
   known as 'delhivery_cleaned_with_chokes.csv' 
   then we checked for any null present
'''
df=pd.read_csv('delhivery_cleaned_1.csv')

In [4]:
df.isnull().sum()

data                                  0
trip_creation_time                    0
route_schedule_uuid                   0
route_type                            0
trip_uuid                             0
source_center                         0
source_name                           0
destination_center                    0
destination_name                      0
od_start_time                         0
od_end_time                           0
start_scan_to_end_scan                0
is_cutoff                             0
cutoff_factor                         0
cutoff_timestamp                      0
actual_distance_to_destination        0
actual_time                           0
osrm_time                             0
osrm_distance                         0
factor                                0
segment_actual_time                   0
segment_osrm_time                     0
segment_osrm_distance                 0
segment_factor                        0
log_segment_factor                    0


In [5]:
''' in selected columns we had only those columns which has high correlation
    with our target columns 'log_segment_actual_time' 
    since our data is already divided into 'training' and 'test'  we creating
    train_df and test_df base on the logic that column df['data'] == 'training'
    data frame column == training for train_df and test_df=='test'
    similiarly  we getting out X_test and y_test , X_train and Y_train 
    X_train,X_test is selected using selected_features 
'''
selected_features = [
    'cutoff_factor',
    'actual_distance_to_destination',
    'osrm_time',
    'osrm_distance',
    'segment_osrm_time',
    'segment_osrm_distance',
]
target_col = 'segment_actual_time'
train_df = df[df['data'] == 'training'].copy()
test_df = df[df['data'] == 'test'].copy()
X_train = train_df[selected_features]
X_test = test_df[selected_features]
y_train = train_df[target_col]
y_test = test_df[target_col]
''' Since train test split is done then
    now we building our model .
    we have used XGBoostRegressor.
    because Unlike Random Forest, which builds trees independently in parallel, XGBoost builds trees sequentially.
    The Mechanism: Each new tree is specifically trained to predict and correct
    the residuals (the errors) made by the previous trees.
    The Logistics Application: Predicting a standard delivery time 
    is relatively easy, but predicting highly volatile edge cases—like a 
    parcel delayed at a specific regional hub during a sudden monsoon storm—is 
    incredibly difficult. XGBoost actively focuses its learning
    capacity on these hard-to-predict, high-error delivery routes, systematically driving down the overall Mean Absolute Error (MAE) of your ETAs.
'''
xgb_model = XGBRegressor(n_estimators=900,learning_rate = 0.10,max_depth=6,subsample=0.8,colsample_bytree=0.8,random_state=42,n_jobs=-1)
xgb_model.fit(X_train,y_train)
y_pred = xgb_model.predict(X_test)
''' we thus then predicted our ETA using first by fitting our 
    training data then by predicting
    rf_model.predict(X_test) 
    we then then predicted
'''
''' we then check our goodness of fit using various metrices like
    mean_absolute_error,mean_squared_error,root_mean_squared_error,
    r2_score and printing them.
'''
mae = mean_absolute_error(y_test,y_pred)
mse = mean_squared_error(y_pred,y_test)
rmse = np.sqrt(mean_squared_error(y_pred,y_test))
r2  = r2_score(y_test,y_pred)
print(y_test.shape,y_pred.shape)
print(f"R² Score: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} minutes")
#Trip level Accuracy.
''' we thus calculated our trip level performnace
    we first created a evaluting DataFrame 
    and then a trip df which contain groupby data based trip id basically to add
    the segment wise data and reset_index for row to be in 0,1,2,3,4....
'''
eval_df = pd.DataFrame({
    'trip_id':test_df['trip_uuid'],
    'actual_time':y_test,
    'predicted_eta':y_pred
})
trip_df = eval_df.groupby('trip_id')[['actual_time','predicted_eta']].sum().reset_index()
trip_df['lower_bound'] = trip_df['actual_time']*0.85
trip_df['upper_bound'] = trip_df['actual_time']*1.15
trip_df['within_15_percent'] = trip_df['predicted_eta'].between(trip_df['lower_bound'],trip_df['upper_bound'])
''' we calculating our percentage of trips that has achieved between
    15_percent and then we finally printing Percentage of trips where 
    ETA is within actual_time ±15%
'''
percentage_accurate_trips = trip_df['within_15_percent'].mean()*100
print("\n--- XGBoost Trip-Level Performance ---")
print(f"Total Unique Trips Evaluated: {len(trip_df)}")
print(f"Percentage of trips where ETA is within actual_time ±15%: {percentage_accurate_trips:.2f}%")

(39376,) (39376,)
R² Score: 0.3022
Mean Absolute Error (MAE): 14.69 minutes
Root Mean Squared Error (RMSE): 46.93 minutes

--- XGBoost Trip-Level Performance ---
Total Unique Trips Evaluated: 4163
Percentage of trips where ETA is within actual_time ±15%: 36.90%


In [ ]:
''' Trip level Peformance is that
    Total_unique trips that are being evaluted is 4163 
    and Percentage of trips where ETA(Expected Time of arrival) is within actual_time ±15%: 37%
    which means that out of 100 trips approximately we predicte 3 of them correctly which is slighly more than
    Random Forest
'''